# 02 · A network that computes: a direction detector

Build a small network that solves a task: in which direction does a dot move across four pixels? The network uses what notebook 01 showed, neurons that fire only when spikes arrive together and delays that make spikes arrive together, and it is written as a weight matrix and a delay matrix, like a layer in a deep network. Everything runs on the Brian2 simulator; participants with cluster access run the network on the SpiNNaker2 chip in section 5 and look at what the chip did.

**Try** marks exercises, **Discuss** marks questions we take up together. Write your answers in the cells below the questions.

The notebook builds on the vendor's tutorials and adds exercises for the workshop. Each section ends with **Go deeper**: where the content comes from and where to read more.

- Vendor tutorials (SpiNNcloud, Apache-2.0): [py-spinnaker2 `tutorials/deep_dive`](https://gitlab.com/spinnaker2/py-spinnaker2/-/tree/v0.8.5/tutorials/deep_dive), referred to as *tutorial 00*, *01*, …
- API documentation: [spinnaker2.gitlab.io/py-spinnaker2](https://spinnaker2.gitlab.io/py-spinnaker2/)
- **Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from matplotlib.animation import FuncAnimation, PillowWriter
from spinnaker2 import brian2_sim, snn

plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

brian2 = brian2_sim.Brian2Backend()  # sections 1 to 4 run on Brian2, section 5 switches to the chip

### The py-spinnaker2 API at a glance

Notebook 01 used each of these objects once. Here they are together, with their options: the reference for building the network below.

| Object | What it is | How it is created |
|---|---|---|
| `snn.Population` | a group of neurons of one model | `snn.Population(size, neuron_model, params, name, record)` |
| `snn.Projection` | the connections from one population to another | `snn.Projection(pre, post, connections)` |
| `snn.Network` | everything that runs together | `snn.Network(name)`, then `net.add(...)` |
| backend | where the network runs | `brian2_sim.Brian2Backend()` or `hardware.SpiNNcloud48NodeBoard()`, then `hw.run(net, steps)` |

A complete program, line by line:

```python
inp = snn.Population(size=2, neuron_model="spike_list",      # an input: sends spikes at given steps
                     params={0: [1, 5], 1: [3]}, name="inp")  # neuron index -> time steps; unlisted neurons stay silent
out = snn.Population(size=1, neuron_model="lif",              # one leaky integrate-and-fire neuron
                     params={"threshold": 10.0, "alpha_decay": 0.9},
                     name="out", record=["spikes", "v"])      # what to record: spikes, v (potential per step)
conns = [[0, 0, 6, 0], [1, 0, 6, 2]]                          # [pre index, post index, weight, delay]
proj = snn.Projection(pre=inp, post=out, connections=conns)
net = snn.Network("example")
net.add(inp, out, proj)                                       # populations and projections
hw.run(net, 20)                                               # 20 time steps; returns nothing
spikes = out.get_spikes()                                     # results are stored in the population
v = out.get_voltages()
```

**Arguments and options**

- `size`: number of neurons. `name`: must be unique within a network.
- `neuron_model`: `"lif"` for the neuron used in the workshop, `"spike_list"` for an input; the other models are listed in the [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html).
- `params` of `lif`: `threshold` (default 1.0), `alpha_decay` (0.8), `i_offset` (0), `v_init` (0), `v_reset` (0) and `reset` (`"reset_by_subtraction"` or `"reset_to_v_reset"`). A number applies to every neuron; a list with one value per neuron sets them individually; `reset` is the same for all. A misspelled name stops with an error.
- `record`: a list with `"spikes"` and/or `"v"`. Only what is recorded can be read back.
- `connections`: one `[pre, post, weight, delay]` per connection; weights of `lif` are whole numbers from −15 to 15, delays from 0 to 7. Several projections into the same population add up.

**What comes back**

- `pop.get_spikes()`: a dict `{neuron index: list of spike times}`. A neuron without spikes may have an empty list or be missing, so use `pop.get_spikes().get(i, [])`.
- `pop.get_voltages()`: a dict `{neuron index: array of v}`, one value per time step, recorded after a possible reset. Only if `"v"` is in `record`.
- `net.reset()`: clears all results. A new run merges its results into the old ones, so reset before running the same network again.

**Go deeper:** tutorial 00, sections 3 and 4 (the four building blocks), tutorial 01 (populations, neuron models and parameters), tutorial 02 (connections); [API docs, snn](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/snn-api.html).

## 1. The input: a dot moving across four pixels

Four pixels in a row, like a tiny strip of an event camera. When a dot passes a pixel, the pixel fires one spike. A dot moving to the right makes pixel 0 fire first, then pixels 1, 2 and 3; a dot moving to the left reverses the order. The animation below shows it in motion. The function `moving_dot` makes this input for a direction and a speed: `step` is the number of time steps the dot needs from one pixel to the next.

In [ ]:
N_PIXELS = 4

def moving_dot(direction, step=1, start=5):
    """Spike times of the pixels, {pixel: [time step]}, for a dot moving "right" or "left"."""
    order = range(N_PIXELS) if direction == "right" else reversed(range(N_PIXELS))
    return {pixel: [start + k * step] for k, pixel in enumerate(order)}


show_trail = True  # animations: True = older events fade out (only for the eye), False = only the events of each step, as the network sees them


def animate_input(inputs, filename, steps=15):
    """Save and show a GIF of inputs {title: spikes}: the pixels in a row, each lighting up when it fires."""
    fig, axes = plt.subplots(1, len(inputs), figsize=(10, 2))
    images, traces = [], []
    for ax, title in zip(axes, inputs):
        images.append(ax.imshow(np.zeros((1, N_PIXELS)), cmap="gray_r", vmin=0, vmax=1))
        ax.set(title=title, xticks=range(N_PIXELS), yticks=[], xlabel="pixel")
        ax.grid(False)
        traces.append(np.zeros(N_PIXELS))
    step_text = fig.suptitle("")

    def update(t):
        for i, spikes in enumerate(inputs.values()):
            fired = np.array([1.0 if t in spikes[p] else 0.0 for p in range(N_PIXELS)])
            traces[i] = (0.6 if show_trail else 0.0) * traces[i] + fired  # fading only for the eye
            images[i].set_data(np.clip(traces[i], 0, 1)[None, :])
        step_text.set_text(f"time step {t}")
        return images

    FuncAnimation(fig, update, frames=steps, interval=400).save(filename, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    display(Image(filename=filename))


animate_input({"dot moving right": moving_dot("right"), "dot moving left": moving_dot("left")}, "detector_input.gif")

The same input as a still picture: a raster plot with one row per pixel and one mark per spike. Read from left to right, the marks climb for a dot moving right and fall for a dot moving left. In both directions every pixel fires exactly once; only the order differs.

In [ ]:
print("dot moving right:", moving_dot("right"))
print("dot moving left: ", moving_dot("left"))

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)  # two small rasters side by side
for ax, direction in zip(axes, ["right", "left"]):
    spikes = moving_dot(direction, step=1)
    ax.eventplot([spikes[p] for p in range(N_PIXELS)], lineoffsets=range(N_PIXELS), linelengths=0.6, colors="black")
    ax.set(title=f"dot moving {direction}", xlabel="time step (1 step = 1 ms)", xlim=(0, 15), yticks=range(N_PIXELS))
axes[0].set_ylabel("pixel")
plt.tight_layout()
plt.show()

## 2. The detector layer: weights and delays as matrices

Two `lif` neurons, one detector for each direction. Each gets one connection from every pixel. The trick is in the delays: for the detector's own direction, they hold the early spikes back so that all four arrive in the same time step. When the dot moves right, pixel 0 fires first, so its spike has to wait longest: delays 3, 2, 1, 0. Which delays does the leftward detector need? Its row in `D` below is still all zeros: fill it in before you run section 3. With weight 3, four spikes arriving together add up to 12, above the threshold of 10.

As in a deep network, the connections of a layer are written as matrices: one row per detector (post), one column per pixel (pre). `W` holds the weights, `D` the delays. A short loop turns them into the connection list `[pre, post, weight, delay]` that `snn.Projection` expects.

In [ ]:
# --- change these ---
W = np.array([[3, 3, 3, 3],   # rightward detector: weights from pixels 0, 1, 2, 3
              [3, 3, 3, 3]])  # leftward detector
D = np.array([[3, 2, 1, 0],   # rightward detector: delays from pixels 0, 1, 2, 3
              [0, 0, 0, 0]])  # leftward detector: fill in the delays from pixels 0, 1, 2, 3
detector_params = {"threshold": 10.0, "alpha_decay": 0.8, "reset": "reset_by_subtraction"}

# --- matrices -> connection list ---
connections = []
for post in range(W.shape[0]):  # every detector ...
    for pre in range(W.shape[1]):  # ... gets a connection from every pixel
        connections.append([pre, post, int(W[post, pre]), int(D[post, pre])])

print(len(connections), "connections; the rightward detector's:", connections[:N_PIXELS])

<details><summary>Solution: the delays of the leftward detector</summary>

When the dot moves left, pixel 3 fires first and pixel 0 last, so pixel 3 has to wait longest: `[0, 1, 2, 3]` (from pixels 0, 1, 2, 3).

</details>

<details><summary>A helper in py-spinnaker2, and why it is not used here</summary>

py-spinnaker2 has a helper for exactly this step, `ann2snn_helpers.connection_list_from_dense_weights(weights, delay)`. It indexes the matrix the other way round (one row per pre neuron) and takes one delay for all connections, which is enough for a layer imported from a trained network but not for this detector, whose delays differ per connection.

</details>

## 3. Run it and read the answer

`run_detector` builds the network for one input (pixels → detectors, with the connections from section 2), runs it and returns the spikes and membrane potentials of the detectors. `show` runs it for several inputs and prints which detector fired: that is the network's answer. If you change `W` or `D`, run the cell of section 2 again first.

In [ ]:
DETECTOR_NAMES = ["rightward", "leftward"]  # one name per row of W


def run_detector(input_spikes, backend=brian2, steps=25):
    """Build the network for one input, run it, return the detectors' spikes and membrane potentials."""
    pixels = snn.Population(size=N_PIXELS, neuron_model="spike_list", params=input_spikes, name="pixels")
    detectors = snn.Population(size=W.shape[0], neuron_model="lif", params=detector_params, name="detectors", record=["spikes", "v"])
    net = snn.Network("direction_detector")
    net.add(pixels, detectors, snn.Projection(pre=pixels, post=detectors, connections=connections))
    backend.run(net, steps)
    return detectors.get_spikes(), detectors.get_voltages()


def show(inputs):
    """Run the detector for each input {title: input_spikes}, print which detectors fired and plot their potentials."""
    fig, axes = plt.subplots(1, len(inputs), figsize=(10, 4), sharey=True, squeeze=False)
    for ax, (title, input_spikes) in zip(axes[0], inputs.items()):
        spikes, v = run_detector(input_spikes)
        fired = [DETECTOR_NAMES[d] for d in range(W.shape[0]) if len(spikes.get(d, [])) > 0]
        print(f"{title}: {', '.join(fired) if fired else 'no detector'} fired")
        for d in range(W.shape[0]):
            color = f"C{d}"  # the same colour for a detector in every panel
            ax.plot(v[d], ".-", color=color, label=DETECTOR_NAMES[d])
            for t in spikes.get(d, []):  # v is recorded after the reset; show the value before it as an open circle
                before = v[d][t] + detector_params["threshold"]  # reset_by_subtraction: before = after + threshold
                ax.plot([t - 1, t], [v[d][t - 1], before], ":", color=color)
                ax.plot(t, before, "o", mfc="white", mec=color, ms=8)
                ax.plot(t, before + 0.8, "v", color=color, ms=10)  # the spike
        ax.axhline(detector_params["threshold"], ls="--", c="0.5")
        ax.set(title=title, xlabel="time step (1 step = 1 ms)")
    axes[0][0].set_ylabel("membrane potential v (threshold = 10)")
    axes[0][0].legend(loc="upper left")
    plt.tight_layout()
    plt.show()


show({"dot moving right": moving_dot("right"), "dot moving left": moving_dot("left")})

The triangles mark the spikes. As in notebook 01, the potential is recorded after the reset; the open circle shows where it was just before, above the threshold.

**See it in motion.** The animation runs the same network for one input and shows, step by step, the pixels firing (left) and the membrane potentials of both detectors (right). Older pixel events fade out only for your eye; the network gets each spike once, and its memory of earlier spikes is the potential of the detectors. Set `show_trail = False` in section 1 to switch the fading off.

In [ ]:
def animate_detector(input_spikes, title, filename):
    """Save and show a GIF: pixels firing (left) and the detectors' membrane potentials building up (right)."""
    steps = 18
    spikes, v = run_detector(input_spikes, steps=steps)
    fired = np.zeros((steps, N_PIXELS))
    for pixel, times in input_spikes.items():
        for t in times:
            fired[t, pixel] = 1

    fig, (ax_px, ax_v) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 2]})
    image = ax_px.imshow(np.zeros((1, N_PIXELS)), cmap="gray_r", vmin=0, vmax=1)
    ax_px.set(xticks=range(N_PIXELS), yticks=[], xlabel="pixel", title=title)
    ax_px.grid(False)
    lines = [ax_v.plot([], [], ".-", color=f"C{d}", label=DETECTOR_NAMES[d])[0] for d in range(W.shape[0])]
    ax_v.axhline(detector_params["threshold"], ls="--", c="0.5")
    ax_v.set(xlim=(0, steps - 1), ylim=(0, 13), xlabel="time step (1 step = 1 ms)", ylabel="membrane potential v")
    ax_v.legend(loc="upper left")
    markers = ax_v.plot([], [], "v", color="k", ms=12)[0]
    trace = np.zeros(N_PIXELS)

    def update(t):
        nonlocal trace
        trace = (0.6 if show_trail else 0.0) * trace + fired[t]  # fading only for the eye
        image.set_data(np.clip(trace, 0, 1)[None, :])
        for d, line in enumerate(lines):
            shown = [v[d][s] + (detector_params["threshold"] if s in spikes.get(d, []) else 0) for s in range(t + 1)]  # before the reset
            line.set_data(range(t + 1), shown)
        done = [(s, detector_params["threshold"] + 1.5) for d in range(W.shape[0]) for s in spikes.get(d, []) if s <= t]
        markers.set_data([s for s, _ in done], [y for _, y in done])
        ax_v.set_title(f"time step {t}")
        return lines + [image, markers]

    FuncAnimation(fig, update, frames=steps, interval=400).save(filename, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    display(Image(filename=filename))


animate_detector(moving_dot("right", step=1), "dot moving right, 1 pixel per step", "detector_right.gif")

## 4. Try it

**Try:** predict the answer first, then run it in the cell below. Write your answers in the cell after it. If time is short, do 1, 2 and 4.

1. The dot moves to the right, but twice as slowly: `moving_dot("right", step=2)`. Which detector fires, and why?
2. Pixel 2 misses the dot: take `moving_dot("right")` and set the spikes of pixel 2 to an empty list. Does the network still give the right answer?
3. *(if time)* Lower the threshold in `detector_params` (section 2) to 8.5 and repeat 1 and 2. What do you gain, and what do you lose?
4. Build a third detector for a dot moving right with `step=2`: add one row to `W`, one to `D` and one name to `DETECTOR_NAMES`. Which delays does it need? Test it with fast and slow dots in both directions.

In [ ]:
# your tries, for example:
slow_right = moving_dot("right", step=2)
show({"dot moving right, slowly": slow_right})

*Your answers:*

1. …
2. …
3. …
4. …

**Discuss:**

- With the dot moving right or left, every pixel fires exactly once, in both directions. Where is the information that tells the two apart?
- The detector layer is a weight matrix, like a layer in a deep network. Could a layer of a deep network that sees only which pixels were active tell the two directions apart? What does the spiking layer have that it lacks?

*Your answers:*

- …
- …

## 5. Run it on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` below and run the cell once per pair, early in the hands-on. A run reserves one of the 6 boards and takes about one minute, mostly for powering up and booting the board; if all boards are busy, it waits until one is free. Participants without cluster access: read along, the expected result is shown in the workshop.

The cell runs the detector for a dot moving right on the chip and on Brian2 and compares the answers. On the chip it also records `time_done`: for every time step, the fraction of the step that the processing element (PE) running the detectors spent computing. The rest of the step it sleeps (the hardware slides, one PE over ten time steps). Brian2 cannot show this: it has no real time.

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

if USE_HARDWARE:
    from spinnaker2 import hardware

    hw = hardware.SpiNNcloud48NodeBoard()
else:
    hw = brian2_sim.Brian2Backend()

print("Running on backend:", type(hw).__name__)

input_spikes = moving_dot("right")
pixels = snn.Population(size=N_PIXELS, neuron_model="spike_list", params=input_spikes, name="pixels")
record = ["spikes", "v", "time_done"] if USE_HARDWARE else ["spikes", "v"]  # time_done only exists on the chip
detectors = snn.Population(size=W.shape[0], neuron_model="lif", params=detector_params, name="detectors", record=record)
net = snn.Network("direction_detector")
net.add(pixels, detectors, snn.Projection(pre=pixels, post=detectors, connections=connections))
hw.run(net, 25)

result = {d: list(ts) for d, ts in detectors.get_spikes().items() if len(ts)}
reference = {d: list(ts) for d, ts in run_detector(input_spikes)[0].items() if len(ts)}  # the same input on Brian2
print("detector spikes:", {DETECTOR_NAMES[d]: ts for d, ts in result.items()})
print("same answer as on Brian2" if result == reference else "different from Brian2")

if USE_HARDWARE:
    for pe, fraction in detectors.get_time_done_times().items():  # one entry per PE that runs detectors
        print(f"{pe}: busy {100 * fraction.mean():.1f} % of each 1 ms step on average, at most {100 * fraction.max():.1f} %")

**Discuss:** the detector PE was busy for a tiny fraction of each step. What would make it busier, and what would happen if the work of one step no longer fit into 1 ms? (Hardware slides: one PE over ten time steps.)

*Your answer:*

- …

## 6. What you built

A network that answers a question about its input: a layer of two `lif` neurons, written as a weight matrix and a delay matrix, that tells the direction of a moving dot from the timing of four spikes. Each detector is a coincidence detector, and the delays decide which timing it waits for: a direction and a speed. On the chip, the same network gives the same answer, while each core sleeps most of every time step.

After the break, a network learns this task itself, on a larger 8 × 8 sensor with noise (notebook 04): it is trained in PyTorch, and the NIR import builds the populations and projections from its weight matrices.

**Go deeper:** delays and delay lines: tutorial 02, section 2; building connection lists from matrices: tutorial 02, section 4; recording and reading what the chip did (`time_done`, profiling): tutorial 05 and the [API docs, profiling](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/profiling.html). Workshop notes: [synapses](../01_knowledge_base/01_primer_snn_basics.md#4-synapses), [what a PE does per time step](../01_knowledge_base/02_hardware_energy.md#2-the-processing-element), [time and memory limits](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).

---

**End of notebook 02.** Then the break; after it, from PyTorch to the chip.